# Experiment 1 — Classification Using Machine Learning Algorithms

## Machine Learning Laboratory — Semester 5

| Field | Details |
|---|---|
| **Roll No** | 3122247001061 |
| **Dataset** | Loan Prediction Dataset (Kaggle) |
| **Target Column** | \Loan_Status\ — Approved (Y) / Rejected (N) |
| **Source** | \dataset/loan_prediction/train.csv\ |
| **Framework** | \src/\ — Reusable across all experiments this semester |

---

### End-to-End Pipeline

`
Load Dataset
    ↓ eda.py
Exploratory Data Analysis
    ↓ preprocessing.py
Preprocessing
    ↓ feature_selection.py
Feature Selection
    ↓ notebook (explicit, visible)
train_test_split()
    ↓ models.py
Model Training
    ↓ evaluation.py
Evaluation
    ↓ observations/
Observations
`

---

### Notebook Role

This notebook is the **orchestration layer only**. 
All reusable logic lives in \src/\. 
This notebook calls one public function per pipeline step and nothing more.


In [ ]:
# =============================================================================
# CELL 2: IMPORTS AND EXPERIMENT CONFIGURATION
# =============================================================================
# This is the ONLY cell that changes when switching to a different dataset.
# All cells below this one are dataset-agnostic.
# =============================================================================

import sys
import os

# -- Python path setup --------------------------------------------------------
# The notebook is at notebooks/. The framework modules are at src/.
# Insert the absolute path to src/ so Python can find them.
sys.path.insert(0, os.path.abspath("../src"))

# -- Standard library imports -------------------------------------------------
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

# -- Framework imports (from src/) --------------------------------------------
# One import per module. All business logic lives inside these modules.
from eda               import classification_eda
from preprocessing     import classification_preprocessing
from feature_selection import classification_feature_selection
from models            import classification_model
from evaluation        import classification_metrics

print("All framework modules imported successfully.")
print()

# =============================================================================
# EXPERIMENT CONFIGURATION
# ---------------------------------------------------------------------------
# To run this notebook on a different dataset, change ONLY the values below.
# Every cell below this one reads from these constants -- never raw values.
#
# NOTE: CLASS_NAMES is NOT defined here.
#       It is determined AUTOMATICALLY in Cell 3 after the dataset is loaded,
#       based on the sorted unique values of TARGET_COLUMN.
#       This keeps the framework fully dataset-agnostic.
# =============================================================================

# -- Dataset ------------------------------------------------------------------
DATASET_PATH    = "../dataset/loan_prediction/train.csv"
TARGET_COLUMN   = "Loan_Status"
EXCLUDE_COLUMNS = ["Loan_ID"]   # Identifier columns -- excluded from features

# -- Feature Selection --------------------------------------------------------
FEATURE_SELECTION_METHOD = "chi2"   # Options: "chi2", "anova", "mutual_info"
K_FEATURES               = 5        # Number of top features to select

# -- Model --------------------------------------------------------------------
MODEL_NAME       = "knn"            # Options: "knn", "decision_tree",
                                    #          "naive_bayes", "logistic_regression",
                                    #          "svm", "random_forest"
MODEL_PARAMETERS = {"n_neighbors": 5}

# -- Train-Test Split ---------------------------------------------------------
TEST_SIZE    = 0.2   # 20% held out for testing
RANDOM_STATE = 42    # Fixed seed for reproducibility across all experiments

# -- Output paths -------------------------------------------------------------
FIGURES_PATH = "../figures/"   # EPS plots saved here by eda.py and evaluation.py
OUTPUT_PATH  = "../output/"    # CSV metrics and TXT reports saved here

# =============================================================================
# CONFIGURATION SUMMARY
# =============================================================================
print("=" * 55)
print("  EXPERIMENT CONFIGURATION")
print("=" * 55)
print(f"  Dataset       : {DATASET_PATH}")
print(f"  Target        : {TARGET_COLUMN}")
print(f"  Exclude       : {EXCLUDE_COLUMNS}")
print( "  Class Names   : determined dynamically in Cell 3")
print(f"  Feature Sel.  : method={FEATURE_SELECTION_METHOD!r},  k={K_FEATURES}")
print(f"  Model         : {MODEL_NAME!r}  params={MODEL_PARAMETERS}")
print(f"  Split         : test_size={TEST_SIZE},  random_state={RANDOM_STATE}")
print(f"  Figures Path  : {FIGURES_PATH}")
print(f"  Output Path   : {OUTPUT_PATH}")
print("=" * 55)


In [ ]:
# =============================================================================
# CELL 3: LOAD AND VALIDATE DATASET
# =============================================================================
# Responsibility  : Load the CSV file and validate it is ready for EDA.
# Does NOT perform: EDA, preprocessing, feature selection, or model training.
# Produces        : df (raw DataFrame), CLASS_NAMES (derived from target values)
# =============================================================================

# -- Guard: check file exists before attempting to load ---------------------
if not os.path.isfile(DATASET_PATH):
    raise FileNotFoundError(
        f"\n[CELL 3 ERROR] Dataset file not found.\n"
        f"  Tried  : {os.path.abspath(DATASET_PATH)}\n"
        f"  Fix    : Update DATASET_PATH in Cell 2 to point to the correct CSV file."
    )

# -- Load ------------------------------------------------------------------
df = pd.read_csv(DATASET_PATH)

# -- Guard: non-empty dataset ----------------------------------------------
if df.empty:
    raise ValueError(
        f"\n[CELL 3 ERROR] The loaded DataFrame is empty.\n"
        f"  File   : {DATASET_PATH}\n"
        f"  Fix    : Verify that the CSV file contains at least one data row."
    )

# -- Guard: target column exists -------------------------------------------
if TARGET_COLUMN not in df.columns:
    raise ValueError(
        f"\n[CELL 3 ERROR] TARGET_COLUMN not found in the dataset.\n"
        f"  Looking for : {TARGET_COLUMN!r}\n"
        f"  Available   : {list(df.columns)}\n"
        f"  Fix         : Update TARGET_COLUMN in Cell 2."
    )

# -- Guard: dataset has at least one feature column ------------------------
if df.shape[1] < 2:
    raise ValueError(
        f"\n[CELL 3 ERROR] Dataset must have at least one feature column\n"
        f"  besides the target column. Found only 1 column total."
    )

# =============================================================================
# DETERMINE CLASS_NAMES DYNAMICALLY
# -----------------------------------------------------------------------------
# CLASS_NAMES are derived from the unique values of TARGET_COLUMN in the raw
# dataset, sorted in ascending order.
#
# Why sorted? LabelEncoder (used in preprocessing.py) assigns integer codes
# in sorted order. So sorted unique values map directly to [0, 1, 2, ...].
#
# Example (Loan Prediction):  ["N", "Y"]  -> N=0, Y=1
# Example (Iris):             ["Iris-setosa", "Iris-versicolor", "Iris-virginica"]
#                             -> setosa=0, versicolor=1, virginica=2
# Example (Diabetes):         ["0", "1"]  -> 0=0, 1=1
#
# This constant is passed to classification_metrics() in Cell 9.
# No other cell needs to modify it.
# =============================================================================
CLASS_NAMES = sorted(df[TARGET_COLUMN].astype(str).unique().tolist())

# =============================================================================
# DATASET SUMMARY
# =============================================================================
print()
print("=" * 55)
print("  DATASET LOADED SUCCESSFULLY")
print("=" * 55)
print(f"  File            : {DATASET_PATH}")
print(f"  Shape           : {df.shape[0]} rows  x  {df.shape[1]} columns")
print(f"  Target Column   : {TARGET_COLUMN!r}")
print(f"  Class Names     : {CLASS_NAMES}  (auto-derived, sorted)")
print()

# -- Missing values summary ------------------------------------------------
missing_counts = df.isnull().sum()
missing_cols   = missing_counts[missing_counts > 0]

if len(missing_cols) == 0:
    print("  Missing Values  : None detected")
else:
    print(f"  Missing Values  : Found in {len(missing_cols)} column(s)")
    for col, cnt in missing_cols.items():
        pct = cnt / len(df) * 100
        print(f"    {col:<25} : {cnt} missing  ({pct:.1f}%)")

print()

# -- Class distribution ----------------------------------------------------
print(f"  Class Distribution  ({TARGET_COLUMN!r}):")
class_counts = df[TARGET_COLUMN].value_counts()
for cls, cnt in class_counts.items():
    pct = cnt / len(df) * 100
    bar = "#" * int(pct / 2)
    print(f"    {str(cls):<10} : {cnt:>4} samples  ({pct:5.1f}%)  {bar}")

print()
print("=" * 55)
print()
print("Next step: Run Cell 4 (EDA).")
print()

# -- Preview the first 5 rows (renders as HTML table in Jupyter) -----------
df.head()
